In [2]:
import pandas as pd
import numpy as np
import os
from ukhls_variables import FILL_STRATEGIES, CATEGORICAL_VARS, CONTINUOUS_VARS, RECODE_MAPS

# CONFIGURATION
PRIMARY_WAVE = "o"        # Wave 15
BACKUP_WAVES = ["n", "m", "l", "k"] # Waves 14 and 13
WAVE_PICKLE_DIR = "../data/2_pickle_ukhls_waves"
OUTPUT_FILE = os.path.join(BASE_PICKLE_DIR, f"{PRIMARY_WAVE}_indresp_backfilled.pkl")

print(f"Targeting Wave {PRIMARY_WAVE} with backups from {BACKUP_WAVES}")

def safe_numeric(series):
    numeric = pd.to_numeric(series, errors='coerce')
    if pd.api.types.is_numeric_dtype(numeric):
        numeric = numeric.replace([np.inf, -np.inf], np.nan)
    return numeric

def get_base_code(col_name, wave_prefix):
    """Strip wave prefix to get the base variable code (e.g. 'o_age_dv' -> 'age_dv')."""
    prefix = f"{wave_prefix}_"
    return col_name[len(prefix):] if col_name.startswith(prefix) else col_name

def impute_column(series, strategy):
    """Apply a named fill strategy to a Series, returning the filled Series."""
    if strategy == "mode":
        fill_val = series.mode(dropna=True)
        return series.fillna(fill_val.iloc[0]) if not fill_val.empty else series
    elif strategy == "median":
        fill_val = series.median(skipna=True)
        return series.fillna(fill_val)
    elif strategy == "zero":
        return series.fillna(0)
    return series  # strategy is None — leave NaN

def build_expanded_master():
    primary_file = os.path.join(BASE_PICKLE_DIR, f"{PRIMARY_WAVE}_indresp_optimized.pkl")
    
    if not os.path.exists(primary_file):
        raise FileNotFoundError(f"Could not find primary wave file: {primary_file}")

    # 1. Load Primary Wave
    print(f"Loading Primary Wave ({PRIMARY_WAVE})...")
    df_master = pd.read_pickle(primary_file).set_index('pidp')
    
    # 2. Iteratively Backfill
    for i, wave in enumerate(BACKUP_WAVES):
        years_to_add = i + 1
        backup_file = os.path.join(BASE_PICKLE_DIR, f"{wave}_indresp_optimized.pkl")
        
        if not os.path.exists(backup_file):
            print(f"Warning: {backup_file} not found. Skipping wave {wave}.")
            continue
            
        print(f"Processing Wave {wave} (Age offset: +{years_to_add})...")
        df_backup = pd.read_pickle(backup_file).set_index('pidp')
        
        # Align column names (e.g., n_age_dv -> o_age_dv)
        df_backup.columns = df_backup.columns.str.replace(f"{wave}_", f"{PRIMARY_WAVE}_")
        
        # A. Pull forward entire missing respondents
        new_ids = df_backup.index.difference(df_master.index)
        if not new_ids.empty:
            df_new = df_backup.loc[new_ids].copy()
            # Age increment logic
            age_col = f"{PRIMARY_WAVE}_age_dv"
            if age_col in df_new.columns:
                age_numeric = safe_numeric(df_new[age_col])
                df_new[age_col] = age_numeric + years_to_add
            
            df_master = pd.concat([df_master, df_new])
            print(f"   -> Added {len(new_ids):,} missing respondents.")

        # B. Patch holes in existing respondents
        # Normalize categorical/int conflicts to object before combine_first
        common_cols = df_master.columns.intersection(df_backup.columns)
        for col in common_cols:
            master_dtype = df_master[col].dtype
            backup_dtype = df_backup[col].dtype
            if isinstance(master_dtype, pd.CategoricalDtype) or isinstance(backup_dtype, pd.CategoricalDtype):
                df_master[col] = df_master[col].astype('object')
                df_backup[col] = df_backup[col].astype('object')

        df_master = df_master.combine_first(df_backup)
        print(f"   -> Patched missing variable values.")

    # 3. Imputation — Apply per-variable fill strategies from ukhls_variables.py
    print("\nApplying fill strategies from ukhls_variables.py...")
    df_master = df_master.reset_index()
    
    imputed = 0
    for col in df_master.columns:
        if 'idp' in col.lower():
            continue  # IDs are handled separately below
        
        base = get_base_code(col, PRIMARY_WAVE)
        strategy = FILL_STRATEGIES.get(base)
        
        # For engineered binary columns not in VARIABLES dict, default to zero fill
        is_binary_derived = any(
            col.endswith(suffix) for suffix in
            ["_drive_to_work", "_work_at_home", "_englang_binary"]
        ) or "_disability_" in col
        
        if strategy is not None:
            missing_before = df_master[col].isna().sum()
            df_master[col] = impute_column(df_master[col], strategy)
            filled = missing_before - df_master[col].isna().sum()
            if filled > 0:
                imputed += filled
        elif is_binary_derived:
            df_master[col] = df_master[col].fillna(0)

    print(f"   -> Filled {imputed:,} missing values using per-variable strategies.")

    # 3.5 Value Recoding — Apply per-variable recode maps from ukhls_variables.py
    print("Applying value recodes from ukhls_variables.py...")
    recoded_cols = 0
    for col in df_master.columns:
        base = get_base_code(col, PRIMARY_WAVE)
        recode = RECODE_MAPS.get(base)
        if recode:
            df_master[col] = df_master[col].replace(recode)
            recoded_cols += 1
    print(f"   -> Recoded {recoded_cols} column(s).")

    # 4. Final Dtype Optimisation — driven by CATEGORICAL_VARS / CONTINUOUS_VARS
    print("Optimizing dtypes...")
    for col in df_master.columns:
        if 'idp' in col.lower():
            id_numeric = safe_numeric(df_master[col]).fillna(0)
            df_master[col] = id_numeric.astype(np.int64)
            continue

        base = get_base_code(col, PRIMARY_WAVE)

        if base in CATEGORICAL_VARS:
            df_master[col] = df_master[col].astype('category')
        elif base in CONTINUOUS_VARS:
            df_master[col] = pd.to_numeric(safe_numeric(df_master[col]), downcast='float')
        elif "_disability_" in col or any(
            col.endswith(s) for s in ["_drive_to_work", "_work_at_home", "_englang_binary"]
        ):
            # Engineered binary columns — store as float32
            df_master[col] = safe_numeric(df_master[col]).astype('float32')
        elif pd.api.types.is_float_dtype(df_master[col]):
            df_master[col] = pd.to_numeric(safe_numeric(df_master[col]), downcast='float')
        elif pd.api.types.is_integer_dtype(df_master[col]):
            df_master[col] = safe_numeric(df_master[col]).astype('Int64')

    # 5. Save as Protocol 5 Pickle (fastest for modern Python)
    df_master.to_pickle(OUTPUT_FILE, protocol=5)
    print(f"DONE. Final Master size: {len(df_master):,} rows.")
    print(f"File saved to: {OUTPUT_FILE}")

build_expanded_master()


Targeting Wave o with backups from ['n', 'm', 'l', 'k']


FileNotFoundError: Could not find primary wave file: ../data/ukhls/pickles/o_indresp_optimized.pkl